# Test an eighty-epoch budget for the lower-rate motion recipe

Notebook 39 was the completed forty-epoch LR0.0003 experiment. Its original local artifact directory is now empty.
The archived report remains historical evidence. This notebook does not recreate or overwrite that run.
Train five fresh MPS folds at LR0.0003, seed 42, with eighty epochs and unchanged patience twelve.
Compare each run’s best checkpoint through epoch forty with its best checkpoint over the full run.
Both selections use the same new trajectory. This is an internal budget control, not a recovered notebook-39 model.
Save predictions every epoch and checkpoints through epoch forty so the selected prefix model remains available.
Keep features, model, loss, optimizer, folds, and final-test reservation fixed. Timo runs training locally.
Read [the protocol](../docs/motion-training-budget.md). Restart the kernel before the first run.


## Locate the checkout and switches

**What:** Resolve the checkout and declare opt-in operations.

**Why:** A report-only run must not prepare or train a model.


In [ ]:
import json
import logging
import os
import shutil
import sys
from pathlib import Path

from IPython import get_ipython
from IPython.display import display

logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)
logger = logging.getLogger("motion_budget_notebook")
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "configs/motion-training-budget-grouped.yaml").exists()),
    Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", Path.cwd() / "sleep-classifier")),
).resolve()
ROOT = Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", str(ROOT))).expanduser().resolve()

REPOSITORY_URL = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
BRANCH = "experiment/motion-training-budget"  # Select the PR branch before merge
RUN_SETUP = False
RUN_PREPARATION = False
RUN_CHECK = False
RUN_TRAINING = False

logger.info("Repository: %s", ROOT)
logger.info("Setup=%s preparation=%s preflight=%s training=%s",
            RUN_SETUP, RUN_PREPARATION, RUN_CHECK, RUN_TRAINING)
PREFIX_EPOCHS = 40


## Optional environment setup

**What:** Optionally clone the PR branch and install locked uv groups.

**Why:** A fresh checkout needs the same package environment and development artifacts.


In [ ]:
if RUN_SETUP:
    if not ROOT.exists():
        !git clone --branch {BRANCH} {REPOSITORY_URL} "{ROOT}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Repository clone failed")
    if shutil.which("uv") is None:
        installer = ROOT / "artifacts/uv-install.sh"
        installer.parent.mkdir(parents=True, exist_ok=True)
        !curl -LsSf https://astral.sh/uv/install.sh -o "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv download failed")
        !sh "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv installation failed")
        os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
    %cd "{ROOT}"
    !uv sync --locked --group gru --group notebook --group dev
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Environment setup failed")
else:
    logger.info("Environment setup skipped")

## Validate the new experiment

**What:** Validate the new config, fold plan, and prefix boundary.

**Why:** The eighty-epoch follow-up must use a fresh directory and require MPS.


In [ ]:
import pandas as pd
import yaml

from sleep_classifier.grouped_validation import GroupedValidationExperiment
from sleep_classifier.neural_training import TrainingConfig
from sleep_classifier.prefix_budget_report import PrefixBudgetReport

experiment = GroupedValidationExperiment(ROOT / "configs/motion-training-budget-grouped.yaml")
plan = experiment.plan()
if (experiment.model_settings["device"] != "mps"
        or experiment.model_settings["epochs"] != 80 or PREFIX_EPOCHS != 40):
    raise ValueError("The follow-up requires explicit MPS and an eighty-versus-forty budget")
if (len(plan["folds"]) != 5 or len(plan["development"]) != 26
        or len(plan["reserved_test"]) != 3 or plan["maximum_epochs"] != 400):
    raise ValueError("The frozen scope or planned budget changed")
if RUN_PREPARATION or RUN_CHECK or RUN_TRAINING:
    if str(TrainingConfig(experiment.model_settings).device()) != "mps":
        raise ValueError("MPS is required without CPU fallback")
reporter = PrefixBudgetReport(experiment.config_path, reference_epochs=PREFIX_EPOCHS)
display(pd.DataFrame(plan["folds"]).set_index("name"))
logger.info("Active config=%s output=%s; preparation exists=%s", experiment.config_path,
            experiment.output_dir, (experiment.output_dir / "manifest.json").exists())


## Verify unchanged optimization controls

**What:** Compare the declared recipe with notebook 39’s config.

**Why:** Only the epoch cap and checkpoint recording may differ; recording does not update the model.


In [ ]:
previous = yaml.safe_load((ROOT / "configs/motion-learning-rate.yaml").read_text())
current = dict(experiment.model_settings)
previous.pop("output_dir")
current.pop("output_dir")
if previous.pop("epochs") != 40 or current.pop("epochs") != 80:
    raise ValueError("Only the declared epoch cap may change")
previous_diagnostics = dict(previous.pop("diagnostics"))
current_diagnostics = dict(current.pop("diagnostics"))
previous_diagnostics.pop("checkpoint_epochs")
if current_diagnostics.pop("checkpoint_epochs") != list(range(1, PREFIX_EPOCHS + 1)):
    raise ValueError("Every potential prefix-selected checkpoint must be saved")
if current != previous or current_diagnostics != previous_diagnostics:
    raise ValueError("Another feature, model, training, or diagnostic setting changed")
if current["learning_rate"] != 0.0003 or current["patience"] != 12 or current["seed"] != 42:
    raise ValueError("Keep LR0.0003, patience twelve, and seed42")
logger.info("Same trajectory controls: raw7/effective9, parameters2900, MPS, patience12")


## Prepare the new folds once

**What:** Optionally create the new prepared manifest and fold inputs.

**Why:** Training requires preparation. Skipping training does not create a manifest.


In [ ]:
if RUN_PREPARATION:
    grouped_config = experiment.config_path
    logger.info("Preparing config=%s output=%s", grouped_config, experiment.output_dir)
    %cd "{ROOT}"
    !uv run --locked --group gru --group notebook python -m sleep_classifier.grouped_validation \
        --config "{grouped_config}" --prepare
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Preparation failed; retain existing evidence and inspect the output")
else:
    logger.info("Preparation skipped; manifest exists=%s",
                (experiment.output_dir / "manifest.json").exists())


## Check shapes without training

**What:** Optionally run no-gradient checks on the prepared folds.

**Why:** A preflight validates model inputs and MPS without optimizer steps.


In [ ]:
if RUN_CHECK:
    experiment.validate_training_start(expected_device="mps")
    %cd "{ROOT}"
    for fold in plan["folds"]:
        fold_config = experiment.output_dir / fold["name"] / "gru.yaml"
        !uv run --locked --group gru --group notebook python -m sleep_classifier.neural_training \
            --config "{fold_config}" --check
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Budget preflight failed")
else:
    logger.info("No-gradient checks skipped")


## Train only after explicit opt-in

**What:** Run the five new eighty-epoch candidate fits.

**Why:** The readiness guard requires a valid manifest, MPS, and fresh model outputs.


In [ ]:
if RUN_TRAINING:
    grouped_config = experiment.config_path
    logger.info("Training config=%s output=%s motion=%s masked_channels=%s", grouped_config,
                experiment.output_dir, experiment.model_settings["motion_features"],
                experiment.model_settings["window_features"].get("masked_channels", []))
    experiment.validate_training_start(expected_device="mps")
    %cd "{ROOT}"
    !uv run --locked --group gru --group notebook python -u -m sleep_classifier.grouped_validation \
        --config "{grouped_config}" --train
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Budget training failed; preserve completed evidence")
else:
    logger.info("Training skipped; no optimizer command was launched")


## Inspect the recorded learning curves

**What:** Plot saved epoch evidence when available.

**Why:** The full trajectory explains convergence, early stopping, and the forty-epoch cutoff.


In [ ]:
import matplotlib.pyplot as plt

from sleep_classifier.experiment_evidence import EpochEvidenceReport

evidence_reader = EpochEvidenceReport(experiment.output_dir)
evidence = evidence_reader.build()
logger.info("Evidence status=%s", evidence["status"])
if any(fold["epochs"] for fold in evidence["folds"]):
    figure = evidence_reader.plot_curves()
    display(figure)
    plt.close(figure)
else:
    logger.info("No completed candidate epochs yet")


## Compare the prefix and full selections

**What:** Build the verified within-trajectory budget report.

**Why:** The missing original notebook-39 files must not be replaced by invented reference predictions.


In [ ]:
result = reporter.build()
logger.info("Budget report status=%s", result["status"])
if result["status"] == "complete":
    fold_rows = [
        {"fold": row["fold"], "completed_epochs": row["completed_epochs"],
         "prefix_best_epoch": row["prefix_best_epoch"], "full_best_epoch": row["full_best_epoch"],
         "prefix_macro_f1": row["reference"]["macro_f1"],
         "full_macro_f1": row["candidate"]["macro_f1"],
         "macro_f1_difference": row["macro_f1_difference"]}
        for row in result["folds"]
    ]
    display(pd.DataFrame(fold_rows).set_index("fold"))
    score_rows = [
        {"budget": role, "mean_fold_macro_f1": result["mean_fold"][role],
         "pooled_macro_f1": result[role]["macro_f1"],
         "accuracy": result[role]["accuracy"], "nll": result[role]["nll"]}
        for role in ("reference", "candidate")
    ]
    display(pd.DataFrame(score_rows).set_index("budget"))
    stage_rows = [
        {"budget": role, "stage": stage, **result[role]["classes"][str(index)]}
        for role in ("reference", "candidate")
        for index, stage in enumerate(("wake", "light", "deep", "REM"))
    ]
    display(pd.DataFrame(stage_rows).set_index(["budget", "stage"]))
    participant_rows = [
        {"participant_id": row["participant_id"],
         "prefix_macro_f1": row["reference"]["macro_f1"],
         "full_macro_f1": row["candidate"]["macro_f1"],
         "macro_f1_difference": row["macro_f1_difference"]}
        for row in result["participants"]
    ]
    display(pd.DataFrame(participant_rows).set_index("participant_id"))
    logger.info("Interpretation: %s; %s", result["control_caveat"], result["selection_caveat"])
    destination = experiment.output_dir / "prefix-budget-comparison.json"
    destination.write_text(json.dumps(result, indent=2) + "\n")
    logger.info("Verified prefix budget comparison saved: %s", destination)
else:
    logger.info("Budget comparison pending; no complete result is written")


## Measured findings and the next decision

Mean fold macro-F1 rises from 0.577675 to 0.582262. Pooled accuracy rises from 66.250% to 66.523%.
The working LR0.001 recipe retains the higher primary score: 0.585343.
Folds 01, 02, and 05 improve. Folds 03 and 04 stop before forty epochs and retain the same selection.
No fold reaches eighty epochs. All show lower training NLL and higher validation NLL after selection.
Selected macro-F1 cannot decrease when the candidate searches a superset of checkpoints. These are descriptive development gains.
Keep LR0.001 with motion. Pause small tuning changes and next audit chronological inference without training.
Read the [measured findings](../docs/experiments/motion-training-budget-2026-11-23/README.md) for fold and stage trade-offs.


## Export a report without training

**What:** Export the notebook from a fresh kernel.

**Why:** The export skips opt-in operations and recursive report generation.


In [ ]:
%cd "{ROOT}"
!uv run --locked --group gru --group notebook python scripts/execute_notebook.py \
    --notebook notebooks/40_motion_training_budget.ipynb
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Budget notebook export failed")
